In [1]:
#d_model=64

In [2]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [3]:
class VimEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                              bidirectional=True, divide_output=True, pscan=True, use_cuda=False)
        self.encoder = VMamba(config)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [4]:
class ROIPatchEmbed3D(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois = n_rois
        self.patch_size = patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed = nn.Embedding(n_rois, d_model)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for emb in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, rois):
        batch_size, n_rois = rois.shape[:2]
        x = rois.reshape(batch_size * n_rois, 1, rois.shape[-3], rois.shape[-2], rois.shape[-1])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(batch_size, n_rois, self.patches_per_roi, self.d_model)
        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, None, :, :] + self.roi_embed.weight[None, :, None, :]
        occupancy = F.max_pool3d((x.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool().reshape(batch_size, n_rois, self.patches_per_roi)
        tokens = tokens.reshape(batch_size, -1, self.d_model)
        valid = valid.reshape(batch_size, -1)
        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)
        return tokens, valid


class VisionMambaBranch(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)

    def forward(self, rois):
        tokens, valid = self.patch_embed(rois)
        tokens = self.vim(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)
        return pooled


class VisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois):
        pooled = self.branch(rois)
        return self.classifier(self.dropout(pooled))

In [5]:
COHORT_CSV     = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_roi64_aug"
CKPT_DIR       = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [6]:
class ROIDataset(Dataset):
    def __init__(self, sessions, labels, cache_dir, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for session_id, label in zip(sessions, labels):
            self.samples.append((session_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        session_id, label, version = self.samples[idx]
        rois = np.array(np.load(f"{self.cache_dir}/{session_id}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long), session_id

In [7]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for rois, labels, _ in loader:
        rois, labels = rois.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(rois), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for rois, labels, _ in loader:
            rois, labels = rois.to(device), labels.to(device)
            out = model(rois)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

In [8]:
BATCH_SIZE = 4
train_loader = DataLoader(ROIDataset(X_train, y_train, MRI_CACHE_AUG, True), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(ROIDataset(X_val, y_val, MRI_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(ROIDataset(X_test, y_test, MRI_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)

In [9]:
print("=== MRI-ONLY, d_model=64, seed 1 only ===")

torch.manual_seed(1); torch.cuda.manual_seed(1); np.random.seed(1); random.seed(1)

model_64 = VisionMambaModel(d_model=64, n_layers=2, n_classes=2, dropout=0.4).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model_64.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

n_params_64 = sum(p.numel() for p in model_64.parameters() if p.requires_grad)
print(f"Params at d_model=64: {n_params_64:,} (vs 44,962 at d_model=32)")

best_val_loss, no_improve, best_epoch = float("inf"), 0, 0
save_path = f"{CKPT_DIR}/vim_dmodel64_mri_seed1.pt"

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8}")
for epoch in range(1, 101):
    train_loss = train_epoch(model_64, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_tpr, val_tnr = evaluate(model_64, val_loader, criterion, device)
    scheduler.step(val_loss)
    print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f}")

    if val_loss < best_val_loss:
        best_val_loss, best_epoch, no_improve = val_loss, epoch, 0
        torch.save(model_64.state_dict(), save_path)
    else:
        no_improve += 1
        if no_improve >= 15:
            print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
            break

model_64.load_state_dict(torch.load(save_path, weights_only=True))
test_loss, test_acc, test_tpr, test_tnr = evaluate(model_64, test_loader, criterion, device)
print(f"\nd_model=64 TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}%")

=== MRI-ONLY, d_model=64, seed 1 only ===
Params at d_model=64: 116,546 (vs 44,962 at d_model=32)
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR
     1 |     0.7056 |     0.6977 |   0.5000 |   1.0000 |   0.0000
     2 |     0.7017 |     0.6902 |   0.5476 |   0.2381 |   0.8571
     3 |     0.6954 |     0.6913 |   0.5000 |   1.0000 |   0.0000
     4 |     0.6971 |     0.6867 |   0.5714 |   0.6667 |   0.4762
     5 |     0.6886 |     0.6859 |   0.5714 |   0.9524 |   0.1905
     6 |     0.6873 |     0.6853 |   0.5714 |   0.9048 |   0.2381
     7 |     0.6878 |     0.6838 |   0.5714 |   0.9524 |   0.1905
     8 |     0.6877 |     0.6817 |   0.6190 |   0.8571 |   0.3810
     9 |     0.6822 |     0.6803 |   0.6190 |   0.3333 |   0.9048
    10 |     0.6816 |     0.6833 |   0.5238 |   0.0952 |   0.9524
    11 |     0.6821 |     0.6767 |   0.5714 |   0.2381 |   0.9048
    12 |     0.6708 |     0.6838 |   0.5238 |   0.0952 |   0.9524
    13 |     0.6724 |     0.6713 |   0.6667 